In [1]:
"""
Wine Quality & Chemical Properties Analysis
Dataset: UCI Wine Quality (red + white, Cortez et al. 2009)

Run:   pip install pandas seaborn matplotlib
       python wine_quality_analysis.py
Output: PNG charts + wine_combined.csv in ./outputs
If you already downloaded winequality-red.csv / winequality-white.csv,
put them next to this script and they are used instead of downloading.
"""
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

BASE = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/"
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

FEATURES = ["fixed acidity", "volatile acidity", "citric acid", "residual sugar",
            "chlorides", "free sulfur dioxide", "total sulfur dioxide",
            "density", "pH", "sulphates", "alcohol"]
RATING_ORDER = ["poor", "average", "good"]
RATING_COLORS = {"poor": "#C4552F", "average": "#8E8A97", "good": "#1F7A6B"}
TYPE_COLORS = {"red": "#9B2D45", "white": "#B8962E"}

sns.set_theme(style="whitegrid", context="notebook")


# ---------------------------------------------------------------- 1. load
def load_wine() -> pd.DataFrame:
    frames = []
    for kind in ("red", "white"):
        name = f"winequality-{kind}.csv"
        source = name if Path(name).exists() else BASE + name
        df = pd.read_csv(source, sep=";")
        df["type"] = kind
        frames.append(df)
    wine = pd.concat(frames, ignore_index=True)

    # poor: 3-5, average: 6, good: 7-9  (the usual split for this dataset)
    wine["rating"] = pd.cut(wine["quality"], bins=[0, 5, 6, 10],
                            labels=RATING_ORDER)
    return wine


# ------------------------------------------------------------- 2. pair plot
def pair_plot(wine: pd.DataFrame, n_sample: int = 1500) -> None:
    """Alcohol & acidity vs. rating. Sampled so the plot stays readable."""
    cols = ["alcohol", "volatile acidity", "fixed acidity", "pH", "quality"]
    sample = wine.sample(min(n_sample, len(wine)), random_state=42)
    g = sns.pairplot(sample, vars=cols, hue="rating", hue_order=RATING_ORDER,
                     palette=RATING_COLORS, diag_kind="kde", corner=False,
                     plot_kws=dict(alpha=0.45, s=18, linewidth=0), height=2.2)
    g.figure.suptitle("Alcohol and acidity against wine rating", y=1.02)
    g.savefig(OUT / "pairplot_alcohol_acidity.png", dpi=150, bbox_inches="tight")
    plt.close("all")


# ---------------------------------------------------------------- 3. boxes
def box_plots(wine: pd.DataFrame) -> None:
    """One box plot per chemical property, split by poor/average/good."""
    fig, axes = plt.subplots(3, 4, figsize=(16, 11))
    for ax, feat in zip(axes.ravel(), FEATURES):
        sns.boxplot(data=wine, x="rating", y=feat, order=RATING_ORDER,
                    palette=RATING_COLORS, hue="rating", legend=False,
                    fliersize=2, linewidth=1, ax=ax)
        ax.set_title(feat)
        ax.set_xlabel("")
    axes.ravel()[-1].axis("off")
    fig.suptitle("Chemical distribution by rating", fontsize=15, y=1.0)
    fig.tight_layout()
    fig.savefig(OUT / "boxplots_by_rating.png", dpi=150, bbox_inches="tight")
    plt.close(fig)


# ------------------------------------------------------------- 4. lm plots
def lm_plot(wine: pd.DataFrame, x: str = "pH") -> None:
    """Linear fit of quality on a feature, one line per wine type."""
    g = sns.lmplot(data=wine, x=x, y="quality", hue="type",
                   palette=TYPE_COLORS, height=5, aspect=1.4,
                   y_jitter=0.2, scatter_kws=dict(alpha=0.25, s=14),
                   line_kws=dict(linewidth=2.5))
    g.figure.suptitle(f"{x} vs. quality", y=1.02)
    g.savefig(OUT / f"lmplot_{x.replace(' ', '_')}_quality.png",
              dpi=150, bbox_inches="tight")
    plt.close("all")


def regression_table(wine: pd.DataFrame) -> pd.DataFrame:
    """Pearson r, slope and R^2 of quality against every feature, per type."""
    rows = []
    for kind, part in wine.groupby("type"):
        for feat in FEATURES:
            r = part[feat].corr(part["quality"])
            slope = r * part["quality"].std() / part[feat].std()
            rows.append(dict(type=kind, feature=feat, r=r, slope=slope, r2=r * r))
    return (pd.DataFrame(rows)
            .sort_values(["type", "r"], key=lambda s: s if s.dtype == object else s.abs(),
                         ascending=[True, False]))


# -------------------------------------------------------------------- main
if __name__ == "__main__":
    wine = load_wine()
    wine.to_csv(OUT / "wine_combined.csv", index=False)
    print(wine.groupby(["type", "rating"], observed=True).size().unstack())

    pair_plot(wine)
    box_plots(wine)
    lm_plot(wine, "pH")
    lm_plot(wine, "alcohol")          # strongest relationship, for contrast

    table = regression_table(wine)
    table.to_csv(OUT / "quality_regression_summary.csv", index=False)
    print("\nquality vs pH:")
    print(table[table.feature == "pH"].round(3).to_string(index=False))
    print(f"\nCharts saved in {OUT.resolve()}")


rating  poor  average  good
type                       
red      744      638   217
white   1640     2198  1060

quality vs pH:
 type feature      r  slope    r2
  red      pH -0.058 -0.302 0.003
white      pH  0.099  0.583 0.010

Charts saved in /content/outputs
